# Lesson 02 - Explorando o Microsoft Agent Framework

O **Microsoft Agent Framework (MAF)** é uma estrutura unificada para construir agentes de IA. Ele fornece uma arquitetura limpa e composicional com quatro blocos de construção principais:

- **Client** – conecta-se a um endpoint de modelo de IA e gerencia a comunicação
- **Agent** – envolve um cliente com instruções e definições de ferramentas
- **Tools** – estendem as capacidades do agente com funções personalizadas que o modelo pode chamar
- **Session** – mantém o histórico da conversa para interações de múltiplas etapas

Nesta lição, construiremos um **agente de reserva de viagens** que verifica a disponibilidade do destino usando esses conceitos.


## Configuração


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Entendendo a Arquitetura do Framework Agent

O Microsoft Agent Framework segue uma arquitetura em camadas:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – Um `AzureAIProjectAgentProvider` conecta-se a uma implantação Azure OpenAI. Ele gerencia a autenticação, formatação de solicitações e análise de respostas.
2. **Agent** – Criado a partir do client via `provider.create_agent()`, o agent combina o acesso ao modelo com instruções (prompt do sistema) e ferramentas.
3. **Tools** – Funções Python decoradas com `@tool` que o agent pode invocar para executar ações ou recuperar dados.
4. **Session** – Um objeto `AgentSession` (criado via `agent.create_session()`) que armazena o histórico da conversa, permitindo diálogos de múltiplas interações onde o agent lembra do contexto anterior.

Vamos construir cada camada passo a passo.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Adicionando Ferramentas com o Decorador @tool

As ferramentas permitem que agentes realizem ações além de gerar texto. O decorador `@tool` converte uma função Python comum em algo que o agente pode chamar.

Pontos principais:
- Use `Annotated[type, "description"]` para que o modelo compreenda cada parâmetro.
- A docstring se torna a descrição da ferramenta que o modelo vê.
- `approval_mode="never_require"` significa que a ferramenta é executada automaticamente, sem confirmação do usuário.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Criando um Agente com Ferramentas

Agora combinamos o cliente, as instruções e as ferramentas em um agente. As `instructions` atuam como o prompt do sistema — elas definem a persona e o comportamento do agente.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Conversas Multi-Turn com Sessões

Uma `AgentSession` (criada via `agent.create_session()`) mantém o registro de todas as mensagens em uma conversa. Ao passar a mesma sessão para cada chamada `agent.run()`, o agente tem acesso ao histórico completo da conversa e pode referir-se a mensagens anteriores.

Passamos `tools=[check_destination_availability]` para que o agente possa chamar nosso verificador de disponibilidade durante cada turno.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Resumo

Nesta lição você explorou os quatro pilares do Microsoft Agent Framework:

| Conceito | O Que Você Aprendeu |
|---------|------------------|
| **Cliente** | `AzureAIProjectAgentProvider` conecta-se ao Azure OpenAI com autenticação baseada em credenciais |
| **Agente** | `provider.create_agent()` agrupa uma conexão de modelo com instruções e um nome |
| **Ferramentas** | O decorador `@tool` expõe funções Python para o agente chamar |
| **Sessão** | `agent.create_session()` mantém o histórico da conversa através de múltiplas interações |

Esses blocos de construção se combinam para criar agentes que podem manter conversas naturais, chamar funções externas e preservar contexto — a base para padrões agentes mais avançados em lições futuras.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:  
Este documento foi traduzido utilizando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos para garantir a precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
